# Event Study explicat pas cu pas

Acesta este primul notebook care trebuie citit.

Scopul lui este sa explice exact logica din codul actual din proiect:

- luam preturi de inchidere
- le transformam in randamente
- invatam un randament zilnic normal din istoricul anterior
- comparam cu randamentele din jurul unui eveniment
- verificam statistic daca abaterea pare reala sau doar zgomot

In acest notebook folosim un esantion local de date reale deja salvat in repo, ca sa poata rula usor.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

plt.style.use("ggplot")
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


def find_sample_path():
    candidates = [
        Path("playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
        Path("../playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
        Path("../../playin-around-with-yahoo-finance-just-testin/yahoo-finance-outputs/history.jsonc"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("Nu am gasit fisierul history.jsonc cu esantionul local de date.")


## 1. Cum arata datele brute

Codul din proiect pleaca de la o serie de preturi `Close`.

Forma scurta din `main.py` este:

```python
import yfinance as yf

data = yf.download("TLV.RO", start="2005-01-01", end="2026-01-01")["Close"]
data = data.squeeze()
```

Mai jos incarcam un esantion local deja salvat, ca sa vedem exact structura datelor.


In [ ]:
sample_path = find_sample_path()

with sample_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

history_df = pd.DataFrame(payload["data"], columns=payload["columns"])
history_df.index = pd.to_datetime(payload["index"], utc=True).tz_convert(None)
history_df.index.name = "Date"

display(history_df.head(8))
print(f"Numar observatii: {len(history_df)}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(history_df.index, history_df["Close"], marker="o", linewidth=2, color="#1f77b4")
ax.set_title("Preturile de inchidere din esantionul local")
ax.set_xlabel("Data")
ax.set_ylabel("Pret Close")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 2. Preturi vs. randamente

Pretul este valoarea activului intr-o zi.
Randamentul spune cu cat s-a schimbat relativ fata de ziua anterioara.

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1
$$

Unde:

- $P_t$ este pretul in ziua $t$
- $R_t$ este randamentul in ziua $t$

In finante, randamentele sunt mai utile decat preturile brute pentru ca pot fi comparate mai usor intre perioade si intre active.


In [ ]:
prices = history_df["Close"].rename("Pret inchidere")
returns = prices.pct_change().rename("Randament")

display(pd.concat([prices, returns], axis=1).head(10))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(prices.index, prices.values, marker="o", linewidth=2, color="#1f77b4")
axes[0].set_title("Preturi de inchidere")
axes[0].set_ylabel("Pret")

axes[1].bar(returns.index, returns.fillna(0).values * 100, color="#ff7f0e")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Randamente zilnice (%)")
axes[1].set_ylabel("%")
axes[1].set_xlabel("Data")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Diferenta financiara dintre valorile efective si randamente

- valorile efective sunt preturile observate in piata
- `real_returns` sunt randamentele calculate din acele preturi
- `normal_returns` sunt randamentele pe care modelul le considera normale
- `abnormal_returns` sunt diferentele dintre ce s-a intamplat si ce era de asteptat

Exemplu:

- daca pretul urca de la `36` la `38`, cresterea absoluta este `2`
- randamentul este `2 / 36 = 0.0556`, adica aproximativ `5.56%`

Randamentul raspunde mai bine la intrebarea financiara utila: "cu cat la suta s-a miscat activul?".


## 3. Fereastra de estimare si fereastra de eveniment

In `main.py`, istoricul este impartit in doua:

- o **fereastra de estimare** pentru invatarea comportamentului normal
- o **fereastra de eveniment** pentru verificarea unei reactii speciale

Mai jos folosim o impartire scurta, doar ca exemplul sa fie usor de urmarit vizual.


In [ ]:
historical_prices = prices.iloc[:13]
event_prices = prices.iloc[13:19]

historical_returns = historical_prices.pct_change().dropna()
real_returns = event_prices.pct_change().dropna()

rezumat_ferestre = pd.DataFrame(
    {
        "Start": [historical_prices.index.min(), event_prices.index.min()],
        "Stop": [historical_prices.index.max(), event_prices.index.max()],
        "Numar preturi": [len(historical_prices), len(event_prices)],
        "Numar randamente": [len(historical_returns), len(real_returns)],
    },
    index=["Fereastra de estimare", "Fereastra de eveniment"],
)

display(rezumat_ferestre)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(prices.index, prices.values, marker="o", linewidth=2, color="#1f77b4")
ax.axvspan(historical_prices.index.min(), historical_prices.index.max(), color="#2ca02c", alpha=0.20, label="Fereastra de estimare")
ax.axvspan(event_prices.index.min(), event_prices.index.max(), color="#d62728", alpha=0.18, label="Fereastra de eveniment")
ax.set_title("Impartirea seriilor in doua ferestre")
ax.set_xlabel("Data")
ax.set_ylabel("Pret Close")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Modelul din proiect ca regresie liniara doar cu intercept

Codul actual face exact asta:

```python
self.historical_returns = historical_data.pct_change().dropna()
self.expected_daily_return = self.historical_returns.mean()
self.historical_std = self.historical_returns.std()
```

Matematic, modelul este:

$$
r_t = \alpha + \varepsilon_t
$$

Unde:

- $r_t$ este randamentul din ziua $t$
- $\alpha$ este randamentul zilnic normal
- $\varepsilon_t$ este zgomotul aleator

Acesta este un model de regresie liniara foarte simplu: nu are variabile explicative, doar intercept.

Antrenarea inseamna sa alegem $\alpha$ astfel incat suma erorilor patratice sa fie minima:

$$
\hat{\alpha} = \arg\min_{\alpha} \sum_{t=1}^{n}(r_t - \alpha)^2 = \bar{r}
$$

Adica solutia OLS este chiar media randamentelor istorice.


In [ ]:
exemplu_randamente = pd.Series([0.0100, -0.0050, 0.0150], name="r_t")
alpha_exemplu = exemplu_randamente.mean()

print(f"Randamente exemplu: {list(exemplu_randamente)}")
print(f"Alpha estimat = media = {alpha_exemplu:.6f} ({alpha_exemplu * 100:.3f}%)")


In [ ]:
expected_daily_return = historical_returns.mean()
historical_std = historical_returns.std()

rezumat_model = pd.Series(
    {
        "Randament mediu zilnic estimat": expected_daily_return,
        "Randament mediu zilnic estimat (%)": expected_daily_return * 100,
        "Volatilitate istorica": historical_std,
        "Volatilitate istorica (%)": historical_std * 100,
    }
)

display(rezumat_model)


## 5. Cum se face predictia

In codul proiectului, predictia este:

```python
return np.repeat(self.expected_daily_return, event_window_length)
```

Adica pentru fiecare zi din fereastra de eveniment:

$$
\hat{r}_t = \hat{\alpha}
$$

Modelul nu incearca sa reproduca fiecare oscilatie zilnica. El spune doar ce ar fi fost "normal", pe baza mediei istorice.


In [ ]:
normal_returns = np.repeat(expected_daily_return, len(real_returns))
abnormal_returns = real_returns.to_numpy() - normal_returns

comparatie = pd.DataFrame(
    {
        "Randament real": real_returns.to_numpy(),
        "Randament normal prezis": normal_returns,
        "Randament anormal": abnormal_returns,
    },
    index=real_returns.index,
)

display(comparatie.mul(100).rename(columns=lambda c: f"{c} (%)").round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(real_returns.index, real_returns.values * 100, marker="o", linewidth=2, label="Randament real", color="#1f77b4")
ax.plot(real_returns.index, normal_returns * 100, marker="s", linestyle="--", linewidth=2, label="Randament normal prezis", color="#ff7f0e")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Randament real vs. randament normal prezis")
ax.set_xlabel("Data")
ax.set_ylabel("Randament (%)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Formula centrala pentru Event Study este:

$$
AR_t = r_t - \hat{r}_t
$$

Unde:

- $AR_t$ este randamentul anormal
- $r_t$ este randamentul real observat
- $\hat{r}_t$ este randamentul normal prezis


## 6. `t-test` si `p-value`

Codul din `t_test.py` calculeaza mai intai media randamentelor anormale:

$$
\overline{AR} = \frac{1}{n}\sum_{t=1}^{n} AR_t
$$

Apoi foloseste statistica t:

$$
t = \frac{\overline{AR}}{s / \sqrt{n}}
$$

Unde:

- $s$ este abaterea standard istorica a randamentelor
- $n$ este numarul de zile din fereastra de eveniment

Iar `p-value` este obtinut din distributia t:

$$
p = 2 \cdot (1 - F_t(|t|; n-1))
$$

Interpretare:

- daca `p` este mic, abaterea observata ar fi rara sub ipoteza de "nimic special"
- in cod, pragul este `0.05`
- daca `p < 0.05`, rezultatul este tratat ca semnificativ statistic


In [ ]:
mean_ar = np.mean(abnormal_returns)
n = len(abnormal_returns)
t_stat = mean_ar / (historical_std / np.sqrt(n))
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))
is_significant = p_value < 0.05

rezultat_test = pd.Series(
    {
        "Media randamentelor anormale": mean_ar,
        "Media randamentelor anormale (%)": mean_ar * 100,
        "Numar observatii": n,
        "t-statistic": t_stat,
        "p-value": p_value,
        "Semnificativ la prag de 5%": is_significant,
    }
)

display(rezultat_test)


In [ ]:
car = np.cumsum(abnormal_returns)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].bar(real_returns.index, abnormal_returns * 100, color="#d62728")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Randamente anormale (%)")
axes[0].set_ylabel("%")

axes[1].plot(real_returns.index, car * 100, marker="o", linewidth=2, color="#9467bd")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("CAR: cumulul randamentelor anormale (%)")
axes[1].set_ylabel("%")
axes[1].set_xlabel("Data")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Ce trebuie retinut

Acesta este fluxul exact din proiect:

1. luam preturile de inchidere
2. le transformam in randamente
3. invatam randamentul zilnic normal din istoricul anterior
4. prezicem randamentul normal in fereastra de eveniment
5. calculam randamente anormale
6. folosim `t-test` si `p-value` pentru a vedea daca abaterea pare importanta statistic

Dupa acest notebook, urmatorul pas natural este al doilea notebook, unde inlocuim modelul simplu cu market model-ul standard din finante.
